# SaaS Funnel Analysis

This notebook is written like a business analyst case study. It starts from messy raw data and ends with actionable recommendations for growth and product teams.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)


from dateutil import parser
def mixed_parse(val):
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    if not s:
        return pd.NaT
    for dayfirst in (False, True):
        try:
            return parser.parse(s, dayfirst=dayfirst)
        except Exception:
            continue
    return pd.NaT


## 2. Load data

In [ ]:
leads = pd.read_csv("../data/leads.csv")
events = pd.read_csv("../data/product_events.csv")

print(leads.shape, events.shape)
display(leads.head())
display(events.head())


## 3. Initial audit

In [ ]:
def audit(df, name):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_pct": (df.isna().mean()*100).round(2),
        "n_unique": df.nunique(dropna=False)
    }).sort_values("missing_pct", ascending=False)
    print(f"\n{name} shape:", df.shape)
    return summary

display(audit(leads, "leads"))
display(audit(events, "events"))

print("Duplicate users in leads:", leads["user_id"].duplicated().sum())
print("Duplicate rows in events:", events.duplicated().sum())


## 4. Data cleaning

In [ ]:
# Standardize text
def clean_text(s):
    return (s.astype(str)
              .str.strip()
              .str.replace(r"\s+", " ", regex=True)
              .replace({"None": np.nan, "nan": np.nan, "": np.nan}))

for col in ["acquisition_channel", "region", "device_type", "company_size"]:
    leads[col] = clean_text(leads[col])

for col in ["event_name", "plan_selected", "platform"]:
    events[col] = clean_text(events[col])

channel_map = {
    "paid search": "Paid Search",
    "Paid Search": "Paid Search",
    "email": "Email",
    "Organic": "Organic",
    "Referral": "Referral",
    "Direct": "Direct",
    "Social": "Social",
    "Partner": "Partner",
    "Email": "Email",
}
device_map = {
    "desktop": "Desktop",
    "Desktop": "Desktop",
    "mobile": "Mobile",
    "Mobile": "Mobile",
    "Tablet": "Tablet",
}
leads["acquisition_channel"] = leads["acquisition_channel"].map(channel_map).fillna("Unknown")
leads["device_type"] = leads["device_type"].str.strip().str.title().map(device_map).fillna("Other")
leads["company_size"] = leads["company_size"].replace({"11 - 50": "11-50"}).fillna("Unknown")
leads["region"] = leads["region"].fillna("Unknown")

# Parse dates
leads["lead_created_at"] = leads["lead_created_at"].apply(mixed_parse)
events["event_timestamp"] = events["event_timestamp"].apply(mixed_parse)

# Numeric cleanup
leads["lead_score"] = pd.to_numeric(leads["lead_score"], errors="coerce")
events["order_value_usd"] = pd.to_numeric(events["order_value_usd"], errors="coerce")
events.loc[events["order_value_usd"] < 0, "order_value_usd"] = np.nan

# Drop obvious bad rows
leads = leads.dropna(subset=["user_id", "lead_created_at"]).drop_duplicates(subset=["user_id"], keep="first")
events = events.dropna(subset=["user_id", "event_name", "event_timestamp"]).drop_duplicates()

print(leads.shape, events.shape)


## 5. Rebuild the funnel table

In [ ]:
stage_order = [
    "landing_page_view",
    "signup_started",
    "signup_completed",
    "workspace_created",
    "team_invite_sent",
    "trial_started",
    "subscription_started"
]

events = events[events["event_name"].isin(stage_order)].copy()
events["event_name"] = pd.Categorical(events["event_name"], categories=stage_order, ordered=True)
events = events.sort_values(["user_id", "event_timestamp", "event_name"])

first_stage = (events.groupby(["user_id", "event_name"], observed=True)["event_timestamp"]
                     .min()
                     .unstack())

funnel = leads.merge(first_stage, left_on="user_id", right_index=True, how="left")
display(funnel.head())


## 6. Funnel conversion rates

In [ ]:
funnel_counts = []
prev_count = len(funnel)
for stage in stage_order:
    current_count = funnel[stage].notna().sum()
    funnel_counts.append({
        "stage": stage,
        "users": current_count,
        "stage_conversion_pct": round(current_count / prev_count * 100, 2) if prev_count else np.nan,
        "overall_conversion_pct": round(current_count / len(funnel) * 100, 2)
    })
    prev_count = current_count

funnel_kpi = pd.DataFrame(funnel_counts)
display(funnel_kpi)

plt.bar(funnel_kpi["stage"], funnel_kpi["users"])
plt.xticks(rotation=45, ha="right")
plt.title("Users by Funnel Stage")
plt.tight_layout()
plt.show()


## 7. Segment analysis

In [ ]:
def segment_conversion(df, segment_col, final_stage="subscription_started"):
    out = (df.groupby(segment_col)[final_stage]
             .apply(lambda x: x.notna().mean())
             .sort_values(ascending=False)
             .mul(100)
             .round(2)
             .reset_index(name="paid_conversion_pct"))
    return out

display(segment_conversion(funnel, "acquisition_channel"))
display(segment_conversion(funnel, "device_type"))
display(segment_conversion(funnel, "region"))


## 8. Time-to-convert

In [ ]:
funnel["days_to_signup"] = (funnel["signup_completed"] - funnel["lead_created_at"]).dt.total_seconds()/86400
funnel["days_to_trial"] = (funnel["trial_started"] - funnel["lead_created_at"]).dt.total_seconds()/86400
funnel["days_to_paid"] = (funnel["subscription_started"] - funnel["lead_created_at"]).dt.total_seconds()/86400

timing = funnel[["days_to_signup","days_to_trial","days_to_paid"]].describe().T
display(timing)

funnel["days_to_paid"].dropna().hist(bins=30)
plt.title("Distribution of Days to Paid Conversion")
plt.xlabel("Days")
plt.show()


## 9. Business findings

In [ ]:
summary = {
    "top_channel_for_paid_conversion": segment_conversion(funnel, "acquisition_channel").iloc[0].to_dict(),
    "top_device_for_paid_conversion": segment_conversion(funnel, "device_type").iloc[0].to_dict(),
    "median_days_to_paid": round(funnel["days_to_paid"].median(), 1),
    "overall_paid_conversion_pct": round(funnel["subscription_started"].notna().mean()*100, 2),
}
summary


## 10. Recommendations

Use this section for a polished analyst write-up. Example themes:
1. Prioritize channels with high downstream paid conversion, not just lead volume.
2. Reduce friction between signup and activation for mobile users.
3. Trigger lifecycle campaigns for users who sign up but fail to create a workspace within 48 hours.
4. Build a weekly funnel health dashboard by channel, device, and region.
